# Study 5 on Google Colab

Runs the locked `docs/PREREGISTRATION_STUDY5.md`: 27 models (about 22 A100 hours) plus data building.

**Before you start**
- *Runtime → Change runtime type → A100 GPU.*
- Colab Secrets (🔑), with notebook access on: `HF_TOKEN` and `OPENAI_API_KEY`. **Never paste keys into cells.**
- The owner's Study 5 approval line must be in `AGENTS.md`.

**Every time Colab (re)connects:** run the setup cell, then the step you are on. Finished work is skipped.

**Licence and blinding**
- Text stays in `MyDrive/study5/private_restricted_text_do_not_share`. Never share it.
- Do not open `scores/` or `logs/` there until step 3 finishes.

In [ ]:
# One setup cell: secrets, Drive, repo, packages. Run this after every (re)connect.
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')            # Colab Secret; never printed
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')  # Colab Secret; needed for step 1 only
OUT = '/content/drive/MyDrive/study5'
if not os.path.exists('/content/prompt_injection'):
    os.system('git clone -q -b claude/vigilant-fermi-r2ovyf https://github.com/dannyg26/prompt_injection /content/prompt_injection')
os.chdir('/content/prompt_injection')
!git pull -q origin claude/vigilant-fermi-r2ovyf
!git log --oneline -1
!pip install -q -r requirements-lock.txt
!pip install -q transformers==4.57.1 datasets==4.4.1 python-dotenv==1.0.1 pandas sentencepiece protobuf openai
!pip install -q --no-deps -e .
!nvidia-smi --query-gpu=name --format=csv,noheader

## 1. Pools: builds and paraphrases the data; no training (about 1 hour)

In [ ]:
!PYTHONPATH=src python scripts/run_study5.py --out "$OUT" --stage pools 2>&1 | grep -E '\[study|Error|Traceback' ; echo "EXIT DONE"

## 2. Pilot: one model, prints the time

In [ ]:
!PYTHONPATH=src python scripts/run_study5.py --out "$OUT" --stage pilot 2>&1 | grep -E '\[study|Error|Traceback' ; echo "EXIT DONE"

## 3. All remaining models, then the analysis (about 22 hours; resumable)

In [ ]:
!PYTHONPATH=src python scripts/run_study5.py --out "$OUT" --stage all 2>&1 | grep -E '\[study|Error|Traceback' ; echo "EXIT DONE"

## After it finishes

Download and send back `MyDrive/study5/public/study5_results.json` and `pool_manifest.json`.

Label the two audits in the private folder (`audit_W_blind.csv`, `audit_Opara_blind.csv`) and send only the counts. Results are reported only after all audits are labelled.

In [ ]:
from google.colab import files
files.download(f'{OUT}/public/study5_results.json')
files.download(f'{OUT}/public/pool_manifest.json')